In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import preprocessing
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import categorical_crossentropy

In [ ]:
trainDf = pd.read_csv('train.csv')
trainDf.shape

In [ ]:
trainDf = trainDf.loc[:, trainDf.nunique() != 1]
corr = trainDf.corr()
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
corr[mask] = np.nan
(corr
 .style
 .background_gradient(cmap='coolwarm', axis=None, vmin=-1, vmax=1)
 .highlight_null(null_color='#f1f1f1')  # Color NaNs grey
 .set_precision(2))

In [ ]:
Id = trainDf.pop('Id')
toa = trainDf.pop('type_of_attack')

total_columns = list(trainDf.columns)
number_columns = list(trainDf.select_dtypes('number').columns)
print('number of columns with numbers: ',number_columns)
print('total columns: ',total_columns)

trainDf_X = trainDf[number_columns]

total_columns, number_columns = set(total_columns), set(number_columns)
nan_columns = total_columns - number_columns
print(nan_columns)

protocolType = np.array(trainDf['protocol_type'])
flag  = np.array(trainDf['flag'])
service = np.array(trainDf['service'])



In [ ]:
protocolType = np.reshape(protocolType, (protocolType.shape[0], 1))
flag = np.reshape(flag, (flag.shape[0], 1))
service = np.reshape(service, (service.shape[0], 1))

labels = toa.nunique()

In [ ]:
toa = np.array(toa)
toa = np.reshape(toa, (toa.shape[0],1))

In [ ]:
label_encoder = preprocessing.LabelEncoder()

encoded_toa = label_encoder.fit_transform(toa)
encoded_protocolType = label_encoder.fit_transform(protocolType)
encoded_flag = label_encoder.fit_transform(flag)
encoded_service = label_encoder.fit_transform(service)

In [ ]:

np.unique(encoded_toa), np.unique(encoded_protocolType), np.unique(encoded_flag), np.unique(encoded_service)

In [ ]:
enc = OneHotEncoder(handle_unknown='ignore')
encoded_toa = np.reshape(encoded_toa, (encoded_toa.shape[0],1))
Y = enc.fit_transform(encoded_toa)

In [ ]:
encoded_protocolType = np.reshape(encoded_protocolType, (encoded_protocolType.shape[0],1))
encoded_service = np.reshape(encoded_service, (encoded_service.shape[0],1))
encoded_flag = np.reshape(encoded_flag, (encoded_flag.shape[0],1))

In [ ]:
protocolType = enc.fit_transform(encoded_protocolType)
service = enc.fit_transform(encoded_service)
flag = enc.fit_transform(encoded_flag)

In [ ]:
protocolType.shape, service.shape, flag.shape

In [ ]:
norm = MinMaxScaler()

# transform training data
X_train = norm.fit_transform(trainDf_X)

In [ ]:
pt = protocolType.toarray()
f = flag.toarray()
s = service.toarray()

In [ ]:
pt_f = np.concatenate((pt, f), axis = 1)
x = np.concatenate((pt_f, s), axis = 1)
X = np.concatenate((x, X_train), axis = 1)

In [ ]:
Y = Y.toarray()

In [ ]:
X.shape, Y.shape

In [ ]:
from sklearn.utils import shuffle

X,Y = shuffle(X,Y)

In [ ]:
model = Sequential([
    Dense(units=128, input_shape=(1,101), activation='relu'),
    Dense(units=256, activation='relu'),
    Dense(units=5, activation='softmax')
])

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(x=X, y=Y, validation_split=0.33,  batch_size=1024, epochs=50, verbose=2)

In [ ]:
model.save('NAPmodel')

In [ ]:
from keras.models import load_model
Model = load_model('NAPmodel')

In [ ]:
Model.summary()

In [ ]:
_, train_acc = model.evaluate(X, Y, verbose=0)
#_, test_acc = model.evaluate(testX, testy, verbose=0)
print('Train: %.3f' % (train_acc))
# plot loss during training

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
y_pred = model.predict(X, verbose=1)
y_pred_bool = np.argmax(y_pred, axis=1)

Y
y_pred_bool.shape

In [ ]:
Y_check = np.argmax(Y, axis = 1)

In [ ]:
Y_check

In [ ]:
y_pred_bool

In [ ]:
y_pred = model.predict(X, batch_size=64, verbose=1)
y_pred_bool = np.argmax(y_pred, axis=1)

print(classification_report(Y_check, y_pred_bool))